# Multimodal Deception Detection Pipeline

Strict OpenFace remediation workflow for the capstone project.

Run cells 1 through 16 in order. Official extraction requires a built OpenFace binary in WSL2 Ubuntu when using the strict pipeline.


---
## Cell 1 - Verify Paths and Runtime


In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'pipeline' / 'config.py').is_file():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root containing pipeline/config.py')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.config import (
    DEVICE,
    DOLOS_FEAT_DIR,
    DOLOS_LABEL_CSV,
    DOLOS_VIDEO_DIR,
    MODEL_DIR,
    OPENFACE_BIN,
    OPENFACE_WSL_BIN,
    PROJECT_ROOT as CONFIG_PROJECT_ROOT,
    TRIAL_FEAT_DIR,
    TRIAL_LABEL_CSV,
    TRIAL_VIDEO_DIR,
    XAI_DIR,
)
from pipeline.feature_extraction import (
    OPENFACE_AVAILABLE,
    get_feature_cache_metadata_path,
    is_official_feature_cache,
)

print(f'Project root: {CONFIG_PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Native OpenFace binary: {OPENFACE_BIN}')
print(f'WSL OpenFace target: {OPENFACE_WSL_BIN}')
print(f'OpenFace available in current runtime: {OPENFACE_AVAILABLE}')
print()

for name, video_dir, label_csv, feat_dir in [
    ('DOLOS', DOLOS_VIDEO_DIR, DOLOS_LABEL_CSV, DOLOS_FEAT_DIR),
    ('Real-Life Trial', TRIAL_VIDEO_DIR, TRIAL_LABEL_CSV, TRIAL_FEAT_DIR),
]:
    video_count = len([f for f in os.listdir(video_dir) if f.endswith('.mp4')]) if os.path.isdir(video_dir) else 0
    label_exists = os.path.isfile(label_csv)
    official_cache = is_official_feature_cache(feat_dir)
    metadata_path = get_feature_cache_metadata_path(feat_dir)
    print(f'{name}:')
    print(f'  videos: {video_count}')
    print(f'  labels: {label_exists} -> {label_csv}')
    print(f'  official cache present: {official_cache}')
    print(f'  cache metadata: {metadata_path}')
    print()

print(f'All working directories present: {os.path.isdir(MODEL_DIR) and os.path.isdir(XAI_DIR)}')


---
## Cell 2 - Import Modules


In [ ]:
import json
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

warnings.filterwarnings('ignore')

from pipeline.config import (
    DEVICE,
    DOLOS_FEAT_DIR,
    DOLOS_LABEL_CSV,
    DOLOS_VIDEO_DIR,
    FEAT_NAMES_PATH,
    FEW_SHOT_FRAC,
    MODEL_DIR,
    MODEL_PATH,
    SCALER_PATH,
    TRIAL_FEAT_DIR,
    TRIAL_LABEL_CSV,
    TRIAL_VIDEO_DIR,
    XAI_DIR,
)
from pipeline.dataset import (
    DeceptionDataset,
    load_features,
    load_named_features,
    make_loader,
    prepare_cross_dataset,
    split_and_normalize,
)
from pipeline.evaluate import (
    full_evaluation,
    load_model,
    plot_comparison_chart,
    print_summary_table,
    protocol_A,
    protocol_B,
    save_all_results,
)
from pipeline.feature_extraction import (
    OPENFACE_AVAILABLE,
    build_feature_names,
    extract_audio_features,
    extract_audio_from_video,
    extract_dataset_features,
    extract_video_features,
    get_openface_binary_path,
    is_official_feature_cache,
    run_openface,
)
from pipeline.model import DeceptionDetector, count_parameters
from pipeline.train import finetune_few_shot, plot_training_curves, train
from pipeline.xai import (
    explain_single_sample,
    load_model_and_scaler,
    modality_ablation,
    plot_shap_summary,
    plot_single_sample_explanation,
)

print('All modules imported successfully')
print(f'PyTorch: {torch.__version__}')
print(f'OpenFace available: {OPENFACE_AVAILABLE}')
print(f'OpenFace binary: {get_openface_binary_path()}')
print(f'Feature vector size: {len(build_feature_names())}')
print(f'Model parameters: {count_parameters(DeceptionDetector()):,}')


---
## Cell 3 - OpenFace Smoke Test


In [ ]:
if not OPENFACE_AVAILABLE:
    raise RuntimeError(
        'Strict OpenFace mode is enabled but no native OpenFace binary is available. '
        'Run this notebook from WSL2 Ubuntu after building OpenFace.'
    )

test_videos = [f for f in os.listdir(DOLOS_VIDEO_DIR) if f.endswith('.mp4')]
if not test_videos:
    raise RuntimeError('No DOLOS videos found for the smoke test')

test_video = os.path.join(DOLOS_VIDEO_DIR, test_videos[0])
test_out_dir = os.path.join(DOLOS_FEAT_DIR, 'openface_out')
tmp_wav_dir = os.path.join(DOLOS_FEAT_DIR, 'tmp_wav')

print(f'Testing OpenFace on: {test_video}')
openface_csv = run_openface(test_video, test_out_dir)
if openface_csv is None:
    raise RuntimeError('OpenFace smoke test failed to produce a CSV')

video_features = extract_video_features(openface_csv)
if video_features is None:
    raise RuntimeError('Failed to aggregate OpenFace CSV into video features')

wav_path = extract_audio_from_video(test_video, tmp_wav_dir)
if wav_path is None:
    raise RuntimeError('ffmpeg audio extraction failed during smoke test')

audio_features = extract_audio_features(wav_path)
if audio_features is None:
    raise RuntimeError('Failed to extract audio features during smoke test')

print(f'OpenFace CSV: {openface_csv}')
print(f'Video feature shape: {video_features.shape}')
print(f'Audio feature shape: {audio_features.shape}')


---
## Cell 4 - Extract DOLOS Features


In [ ]:
dolos_force_rebuild = not is_official_feature_cache(DOLOS_FEAT_DIR)
print(f'Official DOLOS cache present: {not dolos_force_rebuild}')
print(f'Force rebuild DOLOS cache: {dolos_force_rebuild}')

dolos_df = extract_dataset_features(
    video_dir=DOLOS_VIDEO_DIR,
    label_csv=DOLOS_LABEL_CSV,
    out_feat_dir=DOLOS_FEAT_DIR,
    dataset_name='DOLOS',
    force_rebuild=dolos_force_rebuild,
)

print(f'DOLOS features shape: {dolos_df.shape}')
print(dolos_df['label'].value_counts().sort_index().to_dict())
dolos_df.head()


---
## Cell 5 - Extract Trial Features


In [ ]:
trial_force_rebuild = not is_official_feature_cache(TRIAL_FEAT_DIR)
print(f'Official Trial cache present: {not trial_force_rebuild}')
print(f'Force rebuild Trial cache: {trial_force_rebuild}')

trial_df = extract_dataset_features(
    video_dir=TRIAL_VIDEO_DIR,
    label_csv=TRIAL_LABEL_CSV,
    out_feat_dir=TRIAL_FEAT_DIR,
    dataset_name='RealLifeTrial',
    force_rebuild=trial_force_rebuild,
)

print(f'Trial features shape: {trial_df.shape}')
print(trial_df['label'].value_counts().sort_index().to_dict())
trial_df.head()


---
## Cell 6 - EDA


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution', fontsize=14, fontweight='bold')

for axis, (name, dataframe) in zip(axes, [('DOLOS', dolos_df), ('Real-Life Trial', trial_df)]):
    counts = dataframe['label'].value_counts().sort_index()
    axis.bar(['Truthful', 'Deceptive'], counts.values, color=['#34A853', '#EA4335'])
    axis.set_title(name)
    axis.set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

sample_features = ['AU01_r_mean', 'AU12_r_mean', 'pose_Rx_mean', 'mfcc1_mean', 'pitch_mean', 'rms_mean']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: DOLOS vs Trial', fontsize=14, fontweight='bold')

for axis, feature_name in zip(axes.flat, sample_features):
    if feature_name in dolos_df.columns and feature_name in trial_df.columns:
        axis.hist(dolos_df[feature_name], bins=30, alpha=0.6, label='DOLOS', color='#4285F4')
        axis.hist(trial_df[feature_name], bins=30, alpha=0.6, label='Trial', color='#EA4335')
        axis.legend(fontsize=8)
    axis.set_title(feature_name)
    axis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Cell 7 - Train on DOLOS


In [ ]:
dolos_feat_csv = os.path.join(DOLOS_FEAT_DIR, 'features.csv')
dolos_model_path = os.path.join(MODEL_DIR, 'deception_model_dolos.pth')
dolos_scaler_path = os.path.join(MODEL_DIR, 'feature_scaler_dolos.pkl')

dolos_results = train(
    feat_csv=dolos_feat_csv,
    run_name='dolos_baseline',
    device_str=DEVICE,
    model_save_path=dolos_model_path,
    scaler_save_path=dolos_scaler_path,
)

shutil.copy2(dolos_model_path, MODEL_PATH)
shutil.copy2(dolos_scaler_path, SCALER_PATH)
print('Primary model and scaler set to the DOLOS baseline artifacts')


---
## Cell 8 - Train on Trial


In [ ]:
trial_feat_csv = os.path.join(TRIAL_FEAT_DIR, 'features.csv')
trial_model_path = os.path.join(MODEL_DIR, 'deception_model_trial.pth')
trial_scaler_path = os.path.join(MODEL_DIR, 'feature_scaler_trial.pkl')

trial_results = train(
    feat_csv=trial_feat_csv,
    run_name='trial_baseline',
    device_str=DEVICE,
    model_save_path=trial_model_path,
    scaler_save_path=trial_scaler_path,
)


---
## Cell 9 - Protocol A


In [ ]:
proto_A1 = protocol_A(
    source_csv=dolos_feat_csv,
    target_csv=trial_feat_csv,
    source_name='DOLOS',
    target_name='Trial',
    model_path=dolos_model_path,
    scaler_path=dolos_scaler_path,
)

proto_A2 = protocol_A(
    source_csv=trial_feat_csv,
    target_csv=dolos_feat_csv,
    source_name='Trial',
    target_name='DOLOS',
    model_path=trial_model_path,
    scaler_path=trial_scaler_path,
)


---
## Cell 10 - Protocol B


In [ ]:
proto_B = protocol_B(
    source_csv=dolos_feat_csv,
    target_csv=trial_feat_csv,
    source_name='DOLOS',
    target_name='Trial',
    few_shot_frac=FEW_SHOT_FRAC,
    model_path=dolos_model_path,
    scaler_path=dolos_scaler_path,
)


---
## Cell 11 - Results Summary


In [ ]:
all_results = [
    {
        'title': 'Protocol C: DOLOS Baseline',
        'acc': dolos_results['test_acc'],
        'f1': dolos_results['test_f1'],
        'auc': dolos_results['test_auc'],
    },
    {
        'title': 'Protocol C: Trial Baseline',
        'acc': trial_results['test_acc'],
        'f1': trial_results['test_f1'],
        'auc': trial_results['test_auc'],
    },
    proto_A1,
    proto_A2,
    proto_B,
]

print_summary_table(all_results)
save_all_results(all_results)
plot_comparison_chart(all_results)


---
## Cell 12 - SHAP Feature Importance


In [ ]:
shap_importance = plot_shap_summary(
    feat_csv=dolos_feat_csv,
    n_background=50,
    n_explain=100,
    model_path=dolos_model_path,
    device_str=DEVICE,
    top_k=20,
)

sorted_shap = sorted(shap_importance.items(), key=lambda item: item[1], reverse=True)
for idx, (name, value) in enumerate(sorted_shap[:10], 1):
    print(f'{idx:2d}. {name:<30} {value:.6f}')


---
## Cell 13 - Modality Ablation


In [ ]:
abl_dolos = modality_ablation(
    feat_csv=dolos_feat_csv,
    ablation_name='DOLOS',
    model_path=dolos_model_path,
)

abl_trial = modality_ablation(
    feat_csv=trial_feat_csv,
    ablation_name='RealLifeTrial',
    model_path=dolos_model_path,
)

print(abl_dolos)
print(abl_trial)


---
## Cell 14 - Single Sample XAI


In [ ]:
model, scaler, feat_names, device = load_model_and_scaler(dolos_model_path, DEVICE)

X_dolos, y_dolos = load_features(dolos_feat_csv)
X_dolos_scaled = scaler.transform(X_dolos)
np.nan_to_num(X_dolos_scaled, copy=False, nan=0.0)

sample_idx = 0
sample_vec = X_dolos_scaled[sample_idx]
true_label = int(y_dolos[sample_idx])

explanation = explain_single_sample(
    feature_vector=sample_vec,
    model=model,
    feat_names=feat_names,
    device=device,
    top_k=10,
)

print(f'True label: {true_label}')
print(f'Prediction: {explanation["prediction"]}')
print(f'Confidence: {explanation["confidence"]:.1%}')
print(f'Dominant modality: {explanation["dominant_modality"]}')
print(explanation['explanation_text'])

plot_single_sample_explanation(explanation)
print(json.dumps(explanation, indent=2))


---
## Cell 15 - Batch Demo on Trial Samples


In [ ]:
X_trial, y_trial = load_features(trial_feat_csv)
X_trial_scaled = scaler.transform(X_trial)
np.nan_to_num(X_trial_scaled, copy=False, nan=0.0)

n_demo = min(20, len(X_trial_scaled))
correct = 0
print(f'Running XAI on {n_demo} Trial samples')
print('-' * 72)

for idx in range(n_demo):
    explanation = explain_single_sample(
        feature_vector=X_trial_scaled[idx],
        model=model,
        feat_names=feat_names,
        device=device,
        top_k=3,
    )
    predicted_class = int(explanation['predicted_class'])
    is_correct = predicted_class == int(y_trial[idx])
    correct += int(is_correct)
    top_feature = explanation['top_features'][0]['name'] if explanation['top_features'] else 'N/A'
    print(
        f'[{idx:02d}] true={int(y_trial[idx])} pred={predicted_class} '
        f'conf={explanation["confidence"]:.1%} modality={explanation["dominant_modality"]} '
        f'top={top_feature} match={is_correct}'
    )

print(f'Batch accuracy: {correct}/{n_demo} = {correct / n_demo:.1%}')


---
## Cell 16 - Save and Verify Artifacts


In [ ]:
feat_names_list = build_feature_names()
with open(FEAT_NAMES_PATH, 'w', encoding='utf-8') as handle:
    json.dump(feat_names_list, handle, indent=2)

artifacts = {
    'Primary model': MODEL_PATH,
    'DOLOS model': dolos_model_path,
    'Trial model': trial_model_path,
    'Feature scaler': SCALER_PATH,
    'DOLOS scaler': dolos_scaler_path,
    'Trial scaler': trial_scaler_path,
    'Feature names': FEAT_NAMES_PATH,
    'Cross-dataset JSON': os.path.join(MODEL_DIR, 'cross_dataset_results.json'),
    'DOLOS features': os.path.join(DOLOS_FEAT_DIR, 'features.csv'),
    'DOLOS metadata': os.path.join(DOLOS_FEAT_DIR, 'features_meta.json'),
    'Trial features': os.path.join(TRIAL_FEAT_DIR, 'features.csv'),
    'Trial metadata': os.path.join(TRIAL_FEAT_DIR, 'features_meta.json'),
    'SHAP summary': os.path.join(XAI_DIR, 'shap_summary.png'),
    'Single sample XAI': os.path.join(XAI_DIR, 'single_sample_xai.png'),
    'Comparison chart': os.path.join(XAI_DIR, 'cross_dataset_summary.png'),
    'DOLOS ablation': os.path.join(XAI_DIR, 'ablation_DOLOS.png'),
    'Trial ablation': os.path.join(XAI_DIR, 'ablation_RealLifeTrial.png'),
    'DOLOS curves': os.path.join(XAI_DIR, 'dolos_baseline_curves.png'),
    'Trial curves': os.path.join(XAI_DIR, 'trial_baseline_curves.png'),
}

print(f'{"Artifact":<24} {"Status":<8} {"Size":>12} Path')
print('-' * 100)
for name, path in artifacts.items():
    exists = os.path.isfile(path)
    size = f'{os.path.getsize(path):,} B' if exists else '-'
    status = 'OK' if exists else 'MISSING'
    print(f'{name:<24} {status:<8} {size:>12} {path}')
